# 02 Retention Analysis

This notebook converts the cleaned transaction layer into customer-level lifecycle, retention, and concentration metrics.

Inputs:

- `../data/processed/clean_transactions.parquet`

Outputs:

- `../data/processed/customer_metrics.parquet`
- `../data/processed/customer_lifecycle_revenue_split.csv`
- `../data/processed/customer_revenue_deciles.csv`
- `../data/processed/retention_matrix.csv`


## Setup

The analysis starts from the processed transaction layer created in Notebook 01. All customer-level KPIs are scoped to identifiable valid order lines.

In [ ]:
from pathlib import Path

import pandas as pd

PROCESSED_DIR = Path("../data/processed")
TRANSACTIONS_PATH = PROCESSED_DIR / "clean_transactions.parquet"

if not TRANSACTIONS_PATH.exists():
    raise FileNotFoundError(
        f"Processed transaction file not found: {TRANSACTIONS_PATH}. "
        "Run notebooks/01_data_cleaning.ipynb first."
    )

df = pd.read_parquet(TRANSACTIONS_PATH)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
df = df.rename(
    columns={
        "invoiceno": "invoice_no",
        "stockcode": "stock_code",
        "invoicedate": "invoice_date",
        "unitprice": "unit_price",
        "customerid": "customer_id",
    }
)

required_columns = {
    "invoice_no",
    "invoice_date",
    "customer_id",
    "is_valid_order_line",
    "analytical_revenue",
}
missing_columns = sorted(required_columns - set(df.columns))

if missing_columns:
    raise ValueError(f"Processed transaction file is missing columns: {missing_columns}")

df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df_valid = df[df["is_valid_order_line"]].copy()

print(f"Processed rows: {len(df):,}")
print(f"Valid customer order lines: {len(df_valid):,}")
df_valid.head()

## Customer Orders

Order lines are grouped to invoice level so customer lifecycle calculations are based on orders rather than individual product rows. Orders with non-positive analytical revenue are excluded from behavioural retention metrics.

In [ ]:
customer_orders = (
    df_valid.groupby(["customer_id", "invoice_no"], as_index=False)
    .agg(
        order_date=("invoice_date", "min"),
        order_revenue=("analytical_revenue", "sum"),
    )
)

customer_orders = customer_orders[customer_orders["order_revenue"] > 0].copy()
customer_orders = customer_orders.sort_values(
    ["customer_id", "order_date", "invoice_no"]
).reset_index(drop=True)

customer_orders["order_rank"] = (
    customer_orders.groupby("customer_id").cumcount() + 1
)
customer_orders["is_first_order"] = customer_orders["order_rank"] == 1
customer_orders["is_repeat_order"] = customer_orders["order_rank"] > 1

print(f"Customer orders: {len(customer_orders):,}")
customer_orders.head()

## Customer Metrics

The customer metrics table is the core customer-level analytical layer. It supports repeat-customer classification, AOV, active customer lifetime, and downstream concentration analysis.

In [ ]:
customer_metrics = (
    customer_orders.groupby("customer_id", as_index=False)
    .agg(
        total_orders=("invoice_no", "nunique"),
        revenue=("order_revenue", "sum"),
        first_purchase=("order_date", "min"),
        last_purchase=("order_date", "max"),
    )
)

customer_metrics["aov"] = (
    customer_metrics["revenue"] / customer_metrics["total_orders"]
)
customer_metrics["active_days"] = (
    customer_metrics["last_purchase"] - customer_metrics["first_purchase"]
).dt.days
customer_metrics["is_repeat_customer"] = customer_metrics["total_orders"] > 1

customer_metrics.to_parquet(
    PROCESSED_DIR / "customer_metrics.parquet",
    index=False,
)

customer_metrics.head()

## Lifecycle Revenue Split

This view separates revenue generated by first orders, repeat orders, one-time customers, and repeat customers. It clarifies whether the business depends more on acquisition volume or retained customer behaviour.

In [ ]:
first_order_revenue = customer_orders.loc[
    customer_orders["is_first_order"],
    "order_revenue",
].sum()
repeat_order_revenue = customer_orders.loc[
    customer_orders["is_repeat_order"],
    "order_revenue",
].sum()

one_time_customers = customer_metrics.loc[
    ~customer_metrics["is_repeat_customer"],
    "customer_id",
]
repeat_customers = customer_metrics.loc[
    customer_metrics["is_repeat_customer"],
    "customer_id",
]

one_time_revenue = customer_orders.loc[
    customer_orders["customer_id"].isin(one_time_customers),
    "order_revenue",
].sum()
repeat_customer_revenue = customer_orders.loc[
    customer_orders["customer_id"].isin(repeat_customers),
    "order_revenue",
].sum()

lifecycle = pd.DataFrame(
    {
        "metric": [
            "first_order_revenue",
            "repeat_order_revenue",
            "one_time_customer_revenue",
            "repeat_customer_revenue",
        ],
        "value": [
            first_order_revenue,
            repeat_order_revenue,
            one_time_revenue,
            repeat_customer_revenue,
        ],
    }
)
lifecycle["share_of_identifiable_revenue"] = (
    lifecycle["value"] / customer_orders["order_revenue"].sum()
)

lifecycle.to_csv(
    PROCESSED_DIR / "customer_lifecycle_revenue_split.csv",
    index=False,
)

lifecycle

## Second Purchase Behaviour

Second purchase conversion and timing show whether customers are returning and how quickly they move from first purchase to repeat behaviour. This is the main bridge from descriptive retention analysis to CRM lifecycle strategy.

In [ ]:
second_orders = customer_orders[customer_orders["order_rank"] == 2].copy()
second_purchase_conversion = len(second_orders) / len(customer_metrics)

days_to_second = second_orders.merge(
    customer_metrics[["customer_id", "first_purchase"]],
    on="customer_id",
    how="left",
)
days_to_second["days_to_second"] = (
    days_to_second["order_date"] - days_to_second["first_purchase"]
).dt.days
median_days_to_second = days_to_second["days_to_second"].median()

second_purchase_summary = pd.DataFrame(
    {
        "metric": [
            "second_purchase_conversion",
            "median_days_to_second_purchase",
        ],
        "value": [
            second_purchase_conversion,
            median_days_to_second,
        ],
    }
)

print(f"Second purchase conversion: {second_purchase_conversion:.2%}")
print(f"Median days to second purchase: {median_days_to_second:.0f}")
second_purchase_summary

## Customer Revenue Concentration

Customers are ranked by revenue and split into deciles. This identifies whether revenue is broadly distributed or dependent on a narrow high-value customer segment.

In [ ]:
customer_metrics = customer_metrics.sort_values("revenue", ascending=False).copy()
customer_metrics["rank"] = range(1, len(customer_metrics) + 1)
customer_metrics["customer_decile"] = pd.qcut(
    customer_metrics["rank"],
    10,
    labels=[
        "Top 10%",
        "10-20%",
        "20-30%",
        "30-40%",
        "40-50%",
        "50-60%",
        "60-70%",
        "70-80%",
        "80-90%",
        "Bottom 10%",
    ],
)

deciles = (
    customer_metrics.groupby("customer_decile", observed=False)
    .agg(
        customers=("customer_id", "count"),
        revenue=("revenue", "sum"),
    )
    .reset_index()
)

total_revenue = customer_metrics["revenue"].sum()
deciles["customer_pct"] = deciles["customers"] / len(customer_metrics)
deciles["revenue_pct"] = deciles["revenue"] / total_revenue

deciles.to_csv(
    PROCESSED_DIR / "customer_revenue_deciles.csv",
    index=False,
)

deciles

## Monthly Retention Cohorts

Each customer is assigned to the month of their first valid order. Retention is measured as the percentage of each acquisition cohort with at least one valid order in subsequent months.

In [ ]:
customer_orders["order_month"] = customer_orders["order_date"].dt.to_period("M")

first_purchase_month = (
    customer_orders.groupby("customer_id")["order_month"]
    .min()
    .rename("cohort_month")
)

customer_orders = customer_orders.merge(
    first_purchase_month,
    on="customer_id",
    how="left",
)
customer_orders["month_index"] = (
    customer_orders["order_month"].astype(int)
    - customer_orders["cohort_month"].astype(int)
)

cohort_counts = (
    customer_orders.groupby(["cohort_month", "month_index"], observed=False)
    .agg(customers=("customer_id", "nunique"))
    .reset_index()
)
cohort_sizes = (
    cohort_counts[cohort_counts["month_index"] == 0]
    [["cohort_month", "customers"]]
    .rename(columns={"customers": "cohort_size"})
)

retention = cohort_counts.merge(cohort_sizes, on="cohort_month", how="left")
retention["retention_pct"] = retention["customers"] / retention["cohort_size"]

retention_matrix = retention.pivot_table(
    index="cohort_month",
    columns="month_index",
    values="retention_pct",
)

retention_matrix.to_csv(PROCESSED_DIR / "retention_matrix.csv")

retention_matrix

## Analytical Summary

The key commercial readout combines revenue dependency, second-purchase timing, and concentration risk.

In [ ]:
summary_metrics = {
    "valid_customers": len(customer_metrics),
    "valid_orders": len(customer_orders),
    "identifiable_revenue": customer_orders["order_revenue"].sum(),
    "repeat_order_revenue_pct": lifecycle.loc[
        lifecycle["metric"] == "repeat_order_revenue",
        "share_of_identifiable_revenue",
    ].iloc[0],
    "repeat_customer_revenue_pct": lifecycle.loc[
        lifecycle["metric"] == "repeat_customer_revenue",
        "share_of_identifiable_revenue",
    ].iloc[0],
    "second_purchase_conversion": second_purchase_conversion,
    "median_days_to_second_purchase": median_days_to_second,
    "top_10_revenue_pct": deciles.loc[
        deciles["customer_decile"] == "Top 10%",
        "revenue_pct",
    ].iloc[0],
}

pd.Series(summary_metrics).to_frame("value")

## Output Validation

The final checks confirm that all downstream output files were created and contain the expected schema.

In [ ]:
expected_outputs = {
    "customer_metrics.parquet": {
        "customer_id",
        "total_orders",
        "revenue",
        "first_purchase",
        "last_purchase",
        "aov",
        "active_days",
        "is_repeat_customer",
    },
    "customer_lifecycle_revenue_split.csv": {
        "metric",
        "value",
        "share_of_identifiable_revenue",
    },
    "customer_revenue_deciles.csv": {
        "customer_decile",
        "customers",
        "revenue",
        "customer_pct",
        "revenue_pct",
    },
    "retention_matrix.csv": set(),
}

for file_name, expected_columns in expected_outputs.items():
    path = PROCESSED_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f"Expected output was not created: {path}")

    if path.suffix == ".parquet":
        output_df = pd.read_parquet(path)
    else:
        output_df = pd.read_csv(path)

    missing_output_columns = sorted(expected_columns - set(output_df.columns))
    if missing_output_columns:
        raise ValueError(
            f"{file_name} is missing columns: {missing_output_columns}"
        )

    print(f"Validated {file_name}: {output_df.shape}")